# Series Fitting: one setup, many datasets

A temperature scan, a kinetic run, a concentration or field series: many
datasets, one model, and the question is how the parameters change along the
series. `fit_series()` fits every dataset on its own with the setup of one
configured fitter, and returns a table of the parameters against the series
variable.

This notebook works through one scan:

1. [A simulated temperature scan](#1.-A-simulated-temperature-scan)
2. [One good fit, then the whole scan](#2.-One-good-fit,-then-the-whole-scan)
3. [Reading the result](#3.-Reading-the-result)
4. [Is the chain trustworthy?](#4.-Is-the-chain-trustworthy?)
5. [Repair a flagged row](#5.-Repair-a-flagged-row)
6. [Save, reopen, export](#6.-Save,-reopen,-export)
7. [Loops, metadata and in-memory datasets](#7.-Loops,-metadata-and-in-memory-datasets)

The data are simulated with a known answer, so every step can be checked:
spheres whose radius grows slowly with temperature and jumps by 10 Å at a
transition at 50 K. One file is broken, and one frame has a radius beyond the
bounds the template allows.

In [ ]:
import os
import tempfile

import numpy as np

from iquana import IQuana, SeriesResult, examples

WORKDIR = tempfile.mkdtemp(prefix='iquana_series_')
SCAN = os.path.join(WORKDIR, 'scan')
os.makedirs(SCAN)
print(WORKDIR)

## 1. A simulated temperature scan

One file per temperature, named `T_<temperature>K.dat`, three columns (Q, I,
dI) as a reduction pipeline might write them.

In [ ]:
TEMPERATURES = list(range(20, 62, 2))


def true_radius(temperature):
    radius = 40.0 + 0.2 * (temperature - 20)
    return radius + 10.0 if temperature >= 50 else radius


for i, temperature in enumerate(TEMPERATURES):
    radius = 95.0 if temperature == 40 else true_radius(temperature)  # one odd frame
    data = examples.simulate(
        'sphere', radius=radius, scale=0.01, background=0.001, noise=0.02, seed=i, npoints=80
    )
    np.savetxt(os.path.join(SCAN, f'T_{temperature:03d}K.dat'), np.column_stack([data.x, data.y, data.dy]))

# The detector software crashed while writing one file.
with open(os.path.join(SCAN, 'T_036K.dat'), 'w') as handle:
    handle.write('# truncated\n')

sorted(os.listdir(SCAN))[:5]

## 2. One good fit, then the whole scan

Set up and fit one dataset exactly as for a single fit. That fitter is then the
**template**: its model, parameters, bounds, resolution, weighting and Q cut
reach every dataset, because each one is fitted by the ordinary `fit()` on a
fresh copy of it. The template itself is not changed.

In [ ]:
fitter = IQuana()
fitter.load_data(os.path.join(SCAN, 'T_020K.dat'))
fitter.set_model('sphere')
fitter.set_param('radius', value=45, min=10, max=80, vary=True)
fitter.set_param('scale', value=0.02, min=1e-4, max=1, vary=True)
fitter.set_param('background', value=0.01, min=0, max=1, vary=True)
fitter.set_resolution('none')   # the simulated files carry no dQ column
fitter.set_q_range(qmin=0.01)   # trim the lowest points; the upper end stays open
fitter.fit();

The `{temperature}` placeholder finds the files, reads the number from each
name, names the series variable and sorts by it. By default each fit starts
from the previous good one (`start='previous'`), which is what a scan whose
parameters drift needs.

While it runs, only the first row, the rows that fail or are flagged, and a
progress line every 10 s are printed. The numbers in front are the table's row
numbers.

In [ ]:
series = fitter.fit_series(os.path.join(SCAN, 'T_{temperature}K.dat'))

## 3. Reading the result

The result renders as a table: the label, the series variable, every varied
parameter with its uncertainty, reduced χ², and the flags. Long tables show the
first and last ten rows; `series.table()` and `series.to_dataframe()` have all
of them.

In [ ]:
series

A **flag** means "look at this row"; it never removes a row or changes a
number. `flagged()` lists the flagged rows with the reason. Row numbers are
identities: they stay the numbers of the original run, here and everywhere.

- `failed`: the file could not be loaded, the Q cut left no points, or the fit
  raised.
- `jump`: a parameter moved much more than is usual in this series (more than
  5 median steps *and* more than 3 combined uncertainties).
- `bound`, `out_of_bounds`, `unconverged`: what the single fit already
  reports.

In [ ]:
series.flagged()

Any row describes itself: where its fit started, its values and its flags.

In [ ]:
series['T_040K.dat']

The trend plot draws every varied parameter, then reduced χ², against the
variable. The jump at 50 K and the odd frame at 40 K stand out; flagged rows are
diamonds that give their reason on hover, and the broken file is marked × on
the bottom axis.

In [ ]:
series.plot()

In [ ]:
series.plot_fits(rows='flagged')

`plot_fits()` names the rows it cannot draw (the broken file) and draws the
rest. Without `rows=` it draws up to nine rows spread over the series.

## 4. Is the chain trustworthy?

Chaining carries each fit's values to the next one, so a fit that is wrong but
looks fine could lead the rest astray. Independent fits from the template's
values (`start='template'`) are the check.

In [ ]:
independent = fitter.fit_series(os.path.join(SCAN, 'T_{temperature}K.dat'), start='template')
chained = {row['label']: row['radius'] for row in series.table()}
[
    row['label']
    for row in independent.table()
    if row['radius'] is not None and abs(row['radius'] - chained[row['label']]) > 0.5
]

An empty list: the two runs agree everywhere, including at the transition.

## 5. Repair a flagged row

The frame at 40 K has a χ²ᵣ in the thousands and a radius pressed against
`max=80`. `series.fitter(row)` opens it as an ordinary `IQuana`, with its data,
setup and result, so everything known from single fits applies.

In [ ]:
odd = series['T_040K.dat']
repair = series.fitter(odd.index)
repair.plot_results()

In [ ]:
repair.set_param('radius', max=150)
repair.fit()
series.update(odd.index, repair)
series[odd.index]

`update()` checks that the fitter holds a fresh result for its current setup,
that its data is the row's data and that its model is the series' model,
before it replaces anything. The row is marked `edited` and the flags of the
whole series are recomputed.

The repaired fit is good, and the frame is still flagged `jump`: its radius
really is far from its neighbours'. Flags point at rows to look at; they do not
say a fit is wrong.

The broken file can be retried once a good copy exists: `series.fitter(row)`
loads the file again, and the same repair path applies.

In [ ]:
broken = series['T_036K.dat']
good = examples.simulate('sphere', radius=true_radius(36), scale=0.01, background=0.001, seed=99, npoints=80)
np.savetxt(os.path.join(SCAN, 'T_036K.dat'), np.column_stack([good.x, good.y, good.dy]))

retry = series.fitter(broken.index)   # the template setup, with the file loaded now
retry.fit()
series.update(broken.index, retry)
series.flagged()

## 6. Save, reopen, export

`save_analysis()` stores each row as a complete single analysis, the same
document `IQuana.save_analysis()` writes. Data files are referenced, not
copied, so a project moved together with its data still finds them. Loading
reads the JSON only: the table and the trend plot need no data files.

In [ ]:
path = os.path.join(WORKDIR, 'scan.json')
series.save_analysis(path)
reopened = SeriesResult.load_analysis(path)
reopened.save_table(os.path.join(WORKDIR, 'scan.csv'))
reopened.to_dataframe().head()

Fitting the trend is left to you; the table gives the numbers. Below the
transition, without the flagged rows:

In [ ]:
rows = [row for row in reopened.table() if not row['flags'] and row['temperature'] < 50]
slope = np.polyfit([r['temperature'] for r in rows], [r['radius'] for r in rows], 1)[0]
print(f'{slope:.3f} Å/K (simulated: 0.200 Å/K)')

## 7. Loops, metadata and in-memory datasets

**Order.** A pattern is sorted by the series value. A hysteresis loop needs the
order of acquisition instead, which is file-name order when the run number
comes first: `order='name'`. Repeated values are fine, and the trend plot draws
the loop as a loop.

In [ ]:
LOOP = os.path.join(WORKDIR, 'loop')
os.makedirs(LOOP)
fields = [0.0, 0.5, 1.0, 0.5, 0.0]
for run, field in enumerate(fields, start=1):
    data = examples.simulate('sphere', radius=40 + 4 * field + (run > 3), scale=0.01, background=0.001, seed=run)
    np.savetxt(os.path.join(LOOP, f'run{run}_H{field}T.dat'), np.column_stack([data.x, data.y, data.dy]))

loop = fitter.fit_series(os.path.join(LOOP, 'run*_H{field}T.dat'), order='name')
loop.plot('radius')

**Values from metadata.** When the files record the value, read it with a
dotted path: `fit_series('loki/run_*.nxs', values='sample.temperature')`. Every
file is read before the first fit; one without a value is an error that lists
the numeric metadata it does have. sasdata has typed fields for temperature,
thickness, transmission, position and orientation; other quantities, such as a
magnetic field, land in `meta_data` under a key that depends on the file.

**In-memory datasets and explicit values.** A list may hold in-memory
datasets (`data_ops` results, simulations), and `values=` pairs a number with
each, in order.

In [ ]:
curves = []
for i, concentration in enumerate([0.5, 1.0, 2.0, 5.0]):
    curve = examples.simulate('sphere', radius=42.0, scale=0.004 * concentration, background=0.001, seed=20 + i)
    curve.title = f'c = {concentration}'
    curves.append(curve)

dilution = IQuana()
dilution.set_model('sphere')
dilution.set_param('radius', value=45, min=10, max=80, vary=True)
dilution.set_param('scale', value=0.01, min=1e-5, max=1, vary=True)
dilution.set_param('background', value=0.001, vary=False)
dilution.set_resolution('none')
dilution.fit_series(curves, values=[0.5, 1.0, 2.0, 5.0], variable='concentration')

## Where next

- `docs/series.md` covers every option: the value sources, order, start values,
  the flag rules, the drill-down table and the file format.
- A series is N independent fits. To fit several datasets as **one** problem,
  with parameters shared between them, see `MultiFitter` and
  `notebooks/simultaneous_fitting.ipynb`.
- This version fits 1D data, one dataset after another. 2D series, parallel
  execution and Bayesian sampling per dataset are planned.